# Data Processing - The Simpsons Dataset

**Author(s):** Pablo Rodríguez Elvira, Adrián Segura Onorato

This notebook performs the initial exploration, cleaning, merging and aggregation of the Simpsons scripts dataset.  
The resulting CSV files are prepared for the interactive visualizations in Altair and Streamlit.

## 1. Load raw datasets

We load the two raw datasets:

- `simpsons_script_lines.csv`: contains the dialogue lines.
- `simpsons_episodes.csv`: contains episode metadata such as season, episode number, title and ratings.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Make pandas output easier to inspect
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path(".")

lines_path = DATA_DIR / "simpsons_script_lines.csv"
episodes_path = DATA_DIR / "simpsons_episodes.csv"

lines = pd.read_csv(lines_path, low_memory=False)
episodes = pd.read_csv(episodes_path, low_memory=False)

print("Script lines shape:", lines.shape)
print("Episodes shape:", episodes.shape)

Script lines shape: (158271, 13)
Episodes shape: (600, 14)


## 2. Exploratory Data Analysis

Before cleaning the data, we inspect the raw files to understand their structure, missing values and possible quality problems.

In [2]:
# First rows of script lines
lines.head()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of both. Sometimes when a disease is in all the magazines and all the new...",848000,true,464,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometimes when a disease is in all the magazines and all the news shows, it's...",no actually it was a little of both sometimes when a disease is in all the magazines and all the news shows its only...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,true,9,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure like to talk to him. He didn't touch my lesson plan. What did he teach ...,856000,true,464,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk to him. He didn't touch my lesson plan. What did he teach you?,i dont know although id sure like to talk to him he didnt touch my lesson plan what did he teach you,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,true,9,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,"Edna Krabappel-Flanders: The polls will be open from now until the end of recess. Now, (SOUR) just in case any of yo...",864000,true,40,3.0,Edna Krabappel-Flanders,Springfield Elementary School,"The polls will be open from now until the end of recess. Now, just in case any of you have decided to put any though...",the polls will be open from now until the end of recess now just in case any of you have decided to put any thought ...,33


In [3]:
# Column names
print("Script lines columns:")
print(lines.columns.tolist())

Script lines columns:
['id', 'episode_id', 'number', 'raw_text', 'timestamp_in_ms', 'speaking_line', 'character_id', 'location_id', 'raw_character_text', 'raw_location_text', 'spoken_words', 'normalized_text', 'word_count']


In [4]:
# Data types and non-null counts
print("Script lines info:")
lines.info()

Script lines info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158271 entries, 0 to 158270
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  158271 non-null  int64  
 1   episode_id          158271 non-null  int64  
 2   number              158271 non-null  int64  
 3   raw_text            158271 non-null  object 
 4   timestamp_in_ms     158271 non-null  object 
 5   speaking_line       158271 non-null  object 
 6   character_id        140750 non-null  object 
 7   location_id         157864 non-null  float64
 8   raw_character_text  140749 non-null  object 
 9   raw_location_text   157863 non-null  object 
 10  spoken_words        132112 non-null  object 
 11  normalized_text     132087 non-null  object 
 12  word_count          132112 non-null  object 
dtypes: float64(1), int64(3), object(9)
memory usage: 15.7+ MB


In [5]:
# Missing values in script lines
missing_lines = lines.isna().sum().sort_values(ascending=False)
missing_lines

normalized_text       26184
spoken_words          26159
word_count            26159
raw_character_text    17522
character_id          17521
raw_location_text       408
location_id             407
timestamp_in_ms           0
episode_id                0
id                        0
raw_text                  0
number                    0
speaking_line             0
dtype: int64

In [6]:
# Distribution of speaking_line values
# We expect this column to indicate whether the row is an actual spoken line.
lines["speaking_line"].value_counts(dropna=False)

speaking_line
true                                     132112
false                                     26158
Guess what. I also play Frankenstein!         1
Name: count, dtype: int64

In [7]:
# Inspect the original word_count column.
# We will not trust it blindly because some values may be malformed or incorrectly parsed.
word_count_numeric = pd.to_numeric(lines["word_count"], errors="coerce")
word_count_numeric.describe()

count    1.320970e+05
mean     4.261253e+01
std      5.245308e+03
min      0.000000e+00
25%      4.000000e+00
50%      8.000000e+00
75%      1.300000e+01
max      1.154000e+06
Name: word_count, dtype: float64

In [8]:
# Check rows where word_count is not numeric
word_count_numeric = pd.to_numeric(lines["word_count"], errors="coerce")

non_numeric_word_count = lines[word_count_numeric.isna() & lines["word_count"].notna()]

print("Rows with non-numeric word_count:", non_numeric_word_count.shape[0])
non_numeric_word_count[[
    "id",
    "episode_id",
    "raw_character_text",
    "spoken_words",
    "word_count"
]].head(10)

Rows with non-numeric word_count: 15


,id,episode_id,raw_character_text,spoken_words,word_count
8082,17667,59,Singers,"IT'S THE FIRST ANNUAL MONTGOMERY BURNS/ AWARD FOR --,its the first annual montgomery burns award for --,9\n17668,59,...",true
71799,81724,282,Revue Cast Members,"WE'RE THE PERFORMERS YOU THOUGHT WERE DEAD / LIKE BONNIE FRANKLIN...AND ADRIAN ZMED...,were the performers you thoug...",true
81136,91095,315,Homer Simpson,"""I-M-O--,i-m-o--,1\n91096,315,285,Homer Simpson: --K"". Get it? ""I am OK.""",true
86744,96708,335,Patty Bouvier,"""Are you a 'Patty' or a 'Selma?' Take our quiz.,are you a patty or a selma take our quiz,10\n96709,335,275,""Captain ...","I'm a Selma."""
115436,125627,445,Moe Szyslak,"""The reason I left you is simple:,the reason i left you is simple,7\n125628,445,217,Marge Simpson: (SHOCKED) ""...I'm...",true
117571,127763,454,Marge Simpson,"""The patrollers were too fast for Eliza and Virgil...,the patrollers were too fast for eliza and virgil,9\n127764,45...","and presto -- you're part of the under-clown railroad! So you got any talent?"""
130608,140906,503,Homer Simpson,"""Override self-destruct protocol with authorization code seven-two-two-five.,override self-destruct protocol with au...","just don't ask me to drive you to the airport."""
152968,4268,14,Thomas Jefferson,"""We hold these truths to be self-evident.,we hold these truths to be self-evident,7\n4357,15,49,(WINFIELD'S HOUSE: E...","the fact that it wasn't me. I've never felt so alive."""
153015,4260,14,Entire Town,"""SLEIGH BELLS RING / ARE YOU LISTENIN'/,sleigh bells ring are you listenin,6\n4261,14,253,Entire Town: IN THE LANE /...",true
154078,5345,18,Tony Bennett,"THERE'S A SWINGIN' TOWN I KNOW /,theres a swingin town i know,6\n5346,18,200,Tony Bennett: CALLED CAPITAL CITY...",true


In [9]:
# Largest original word_count values.
# Extremely high values are suspicious for one dialogue line.
lines.assign(
    word_count_numeric=pd.to_numeric(lines["word_count"], errors="coerce")
).sort_values(
    "word_count_numeric",
    ascending=False
)[[
    "id",
    "episode_id",
    "raw_character_text",
    "spoken_words",
    "word_count",
    "word_count_numeric"
]].head(10)

,id,episode_id,raw_character_text,spoken_words,word_count,word_count_numeric
153016,4263,14,Entire Town,"""GONE AWAY IS THE BLUEBIRD / HERE TO STAY...,gone away is the bluebird here to stay,8\n4264,14,256,Bart Simpson: Got...",1154000,1154000.0
59908,69807,243,Marge Simpson,"""My gold is in the heart of every freedom-loving American.,my gold is in the heart of every freedom-loving american,...",1145000,1145000.0
73537,83465,289,Robert Pinsky,"""Impossible to tell in writing. 'Bashõ',impossible to tell in writing bashõ,6\n83466,289,165,""Robert Pinsky: He name...",672000,672000.0
52605,62483,220,ABBA,"""If you wanna be my lover / You gotta get with my friends / Makin' love forever / Friendship's where that ends / If ...",571000,571000.0
78951,88907,308,Homer Simpson,"""Nausea... cravings... knocked-up feeling... She was pregnant with Bart! And that's the reason she stayed with me.,n...",409000,409000.0
77228,87170,302,Lisa Simpson,"""Molochai desiratum maledictu... nosferatu ascendum corporalis...,molochai desiratum maledictu nosferatu ascendum co...",147000,147000.0
101152,111237,390,Lisa Simpson,"""...Hitachee Tribe.,hitachee tribe,2\n111238,390,17,""Lisa Simpson: Wait",117000,117000.0
154163,5432,19,Bart Simpson,"""One o'clock -- still just a potato.,one oclock -- still just a potato,7\n5433,19,6,""Ned Flanders: (PLEASANTLY) Hey ...",108000,108000.0
13085,22701,76,Grampa Simpson,One trick is to tell them stories that don't go anywhere... Like the time I caught the ferry over to Shelbyville. I ...,122,122.0
124004,134211,478,Randy Newman,"It was not! Who else in here's gonna tip ten dollars? Keith Urban, Helen Mirren? I don't think so. They don't have t...",116,116.0


In [10]:
# Check rows with embedded CSV fragments inside spoken_words.
# These rows are likely parsing errors and should be removed.
bad_rows = lines["spoken_words"].astype(str).str.contains(
    r"\n\d+,\d+,\d+,",
    regex=True,
    na=False
)

print("Rows with possible embedded CSV fragments:", bad_rows.sum())

lines.loc[bad_rows, [
    "id",
    "episode_id",
    "raw_character_text",
    "spoken_words"
]].head()

Rows with possible embedded CSV fragments: 23


,id,episode_id,raw_character_text,spoken_words
8082,17667,59,Singers,"IT'S THE FIRST ANNUAL MONTGOMERY BURNS/ AWARD FOR --,its the first annual montgomery burns award for --,9\n17668,59,..."
52605,62483,220,ABBA,"""If you wanna be my lover / You gotta get with my friends / Makin' love forever / Friendship's where that ends / If ..."
59908,69807,243,Marge Simpson,"""My gold is in the heart of every freedom-loving American.,my gold is in the heart of every freedom-loving american,..."
71799,81724,282,Revue Cast Members,"WE'RE THE PERFORMERS YOU THOUGHT WERE DEAD / LIKE BONNIE FRANKLIN...AND ADRIAN ZMED...,were the performers you thoug..."
73537,83465,289,Robert Pinsky,"""Impossible to tell in writing. 'Bashõ',impossible to tell in writing bashõ,6\n83466,289,165,""Robert Pinsky: He name..."


In [11]:
# Initial number of unique characters.
# This explains why using all characters would make the visualizations too cluttered.
print("Unique raw characters:", lines["raw_character_text"].nunique())

lines["raw_character_text"].value_counts().head(20)

Unique raw characters: 6765


raw_character_text
Homer Simpson              29842
Marge Simpson              14159
Bart Simpson               13777
Lisa Simpson               11502
C. Montgomery Burns         3172
Moe Szyslak                 2864
Seymour Skinner             2443
Ned Flanders                2145
Grampa Simpson              1886
Milhouse Van Houten         1862
Chief Wiggum                1836
Krusty the Clown            1772
Nelson Muntz                1174
Lenny Leonard               1166
Apu Nahasapeemapetilon      1006
Waylon Smithers             1001
Kent Brockman                894
Carl Carlson                 883
Edna Krabappel-Flanders      745
Dr. Julius Hibbert           693
Name: count, dtype: int64

In [12]:
# Check whether episode ids from script lines exist in episodes
line_episode_ids = pd.to_numeric(lines["episode_id"], errors="coerce").dropna().astype(int)
episode_ids = pd.to_numeric(episodes["id"], errors="coerce").dropna().astype(int)

missing_episode_ids = set(line_episode_ids.unique()) - set(episode_ids.unique())

print("Episode IDs in script lines not found in episodes:", len(missing_episode_ids))
print(sorted(list(missing_episode_ids))[:20])

Episode IDs in script lines not found in episodes: 0
[]


## 3. Clean script lines

The raw script file contains non-spoken rows, missing values and some corrupted rows.  
We keep only real spoken lines with valid character and text fields.

In [13]:
# Work on a copy to keep the raw dataframe untouched
lines_clean = lines.copy()

# Normalize speaking_line
lines_clean["speaking_line"] = (
    lines_clean["speaking_line"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep only actual spoken lines
lines_clean = lines_clean[lines_clean["speaking_line"] == "true"].copy()

# Remove corrupted rows with embedded CSV fragments
bad_rows = lines_clean["spoken_words"].astype(str).str.contains(
    r"\n\d+,\d+,\d+,",
    regex=True,
    na=False
)

lines_clean = lines_clean[~bad_rows].copy()

# Drop rows without essential information
lines_clean = lines_clean.dropna(subset=[
    "episode_id",
    "raw_character_text",
    "spoken_words",
    "normalized_text"
])

# Clean timestamp_in_ms: coerce non-numeric values (e.g. 'Springfield Elementary School')
# to NaN, then drop NaN and zero timestamps (no meaningful position in the episode).
lines_clean["timestamp_in_ms"] = pd.to_numeric(lines_clean["timestamp_in_ms"], errors="coerce")
lines_clean = lines_clean.dropna(subset=["timestamp_in_ms"])
lines_clean = lines_clean[lines_clean["timestamp_in_ms"] > 0].copy()

# Clean character names
lines_clean["character"] = (
    lines_clean["raw_character_text"]
    .astype(str)
    .str.strip()
)

# Convert episode_id to numeric
lines_clean["episode_id"] = pd.to_numeric(lines_clean["episode_id"], errors="coerce")
lines_clean = lines_clean.dropna(subset=["episode_id"])
lines_clean["episode_id"] = lines_clean["episode_id"].astype(int)

print("Cleaned script lines shape:", lines_clean.shape)

Cleaned script lines shape: (131697, 14)


## 4. Recompute word count

The original `word_count` column contains suspicious values, so we recompute it from `normalized_text`.

In [14]:
# Recompute word count from normalized_text
lines_clean["word_count"] = (
    lines_clean["normalized_text"]
    .astype(str)
    .str.split()
    .str.len()
)

# Remove empty lines
lines_clean = lines_clean[lines_clean["word_count"] > 0].copy()

lines_clean["word_count"].describe()

count    131697.000000
mean          9.882290
std           8.048518
min           1.000000
25%           4.000000
50%           8.000000
75%          13.000000
max         122.000000
Name: word_count, dtype: float64

## 5. Compute sentence count

We approximate the number of sentences by counting sentence-ending punctuation marks.  
If a spoken line has no `.`, `!` or `?`, we count it as one sentence to avoid zero-sentence dialogue lines.

In [15]:
lines_clean["sentence_count"] = (
    lines_clean["spoken_words"]
    .astype(str)
    .str.count(r"[.!?]+")
)

# If a line has no punctuation, count it as one sentence
lines_clean["sentence_count"] = lines_clean["sentence_count"].clip(lower=1)

lines_clean[["spoken_words", "word_count", "sentence_count"]].head()

,spoken_words,word_count,sentence_count
0,"No, actually, it was a little of both. Sometimes when a disease is in all the magazines and all the news shows, it's...",31,2
1,Where's Mr. Bergstrom?,3,2
2,I don't know. Although I'd sure like to talk to him. He didn't touch my lesson plan. What did he teach you?,22,4
3,That life is worth living.,5,1
4,"The polls will be open from now until the end of recess. Now, just in case any of you have decided to put any though...",33,3


## 6. Clean episodes dataset

We clean the episode identifier and keep the metadata columns needed for the analysis.

In [16]:
episodes_clean = episodes.copy()

# Convert id to numeric
episodes_clean["id"] = pd.to_numeric(episodes_clean["id"], errors="coerce")
episodes_clean = episodes_clean.dropna(subset=["id"])
episodes_clean["id"] = episodes_clean["id"].astype(int)

# Convert season to numeric
episodes_clean["season"] = pd.to_numeric(episodes_clean["season"], errors="coerce")
episodes_clean = episodes_clean.dropna(subset=["season"])
episodes_clean["season"] = episodes_clean["season"].astype(int)

# Keep only seasons 1-25 (complete seasons in the dataset)
episodes_clean = episodes_clean[episodes_clean["season"].between(1, 25)].copy()

print(f"Seasons after filter: {sorted(episodes_clean['season'].unique())}")
print(f"Episodes after filter: {episodes_clean.shape[0]}")

# Keep useful columns (including title for episode-level labels)
episode_cols = ["id", "season", "number_in_season", "number_in_series", "title"]
episodes_small = episodes_clean[episode_cols].copy()

episodes_small.head()

Seasons after filter: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]
Episodes after filter: 552


,id,season,number_in_season,number_in_series,title
0,10,1,10,10,Homer's Night Out
1,12,1,12,12,Krusty Gets Busted
2,14,2,1,14,"Bart Gets an ""F"""
3,17,2,4,17,Two Cars in Every Garage and Three Eyes on Every Fish
4,19,2,6,19,Dead Putting Society


## 7. Merge script lines with episodes

We join the script lines with the episode metadata. Since `episodes_small` already contains only seasons 1–25, the merge implicitly filters the script lines to those seasons — no extra filter step needed.

In [17]:
df = lines_clean.merge(
    episodes_small,
    left_on="episode_id",
    right_on="id",
    how="inner"  # inner: only keep lines whose episode_id exists in seasons 1-25
)

# Convert to int
df["season"] = df["season"].astype(int)
df["number_in_season"] = df["number_in_season"].astype(int)

print("Merged dataset shape:", df.shape)
print("Seasons present:", sorted(df["season"].unique()))

df[["episode_id", "season", "number_in_season", "title", "character", "spoken_words"]].head()

Merged dataset shape: (128325, 20)
Seasons present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]


,episode_id,season,number_in_season,title,character,spoken_words
0,32,2,19,Lisa's Substitute,Miss Hoover,"No, actually, it was a little of both. Sometimes when a disease is in all the magazines and all the news shows, it's..."
1,32,2,19,Lisa's Substitute,Lisa Simpson,Where's Mr. Bergstrom?
2,32,2,19,Lisa's Substitute,Miss Hoover,I don't know. Although I'd sure like to talk to him. He didn't touch my lesson plan. What did he teach you?
3,32,2,19,Lisa's Substitute,Lisa Simpson,That life is worth living.
4,32,2,19,Lisa's Substitute,Edna Krabappel-Flanders,"The polls will be open from now until the end of recess. Now, just in case any of you have decided to put any though..."


## 8. Keep useful columns

We keep only the variables needed for the visual analytics tool. `title` is kept here because it will be useful for episode-level tooltips in the charts.

In [18]:
df_clean = df[[
    "episode_id",
    "season",
    "number_in_season",
    "number_in_series",
    "title",
    "character",
    "spoken_words",
    "normalized_text",
    "word_count",
    "sentence_count",
    "timestamp_in_ms"
]].copy()

print("Clean dataset shape:", df_clean.shape)
print("Seasons:", sorted(df_clean["season"].unique()))
df_clean.head()

Clean dataset shape: (128325, 11)
Seasons: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]


,episode_id,season,number_in_season,number_in_series,title,character,spoken_words,normalized_text,word_count,sentence_count,timestamp_in_ms
0,32,2,19,32,Lisa's Substitute,Miss Hoover,"No, actually, it was a little of both. Sometimes when a disease is in all the magazines and all the news shows, it's...",no actually it was a little of both sometimes when a disease is in all the magazines and all the news shows its only...,31,2,848000
1,32,2,19,32,Lisa's Substitute,Lisa Simpson,Where's Mr. Bergstrom?,wheres mr bergstrom,3,2,856000
2,32,2,19,32,Lisa's Substitute,Miss Hoover,I don't know. Although I'd sure like to talk to him. He didn't touch my lesson plan. What did he teach you?,i dont know although id sure like to talk to him he didnt touch my lesson plan what did he teach you,22,4,856000
3,32,2,19,32,Lisa's Substitute,Lisa Simpson,That life is worth living.,that life is worth living,5,1,864000
4,32,2,19,32,Lisa's Substitute,Edna Krabappel-Flanders,"The polls will be open from now until the end of recess. Now, just in case any of you have decided to put any though...",the polls will be open from now until the end of recess now just in case any of you have decided to put any thought ...,33,3,864000


## 9. Select the top 10 most relevant characters

Since the dataset contains many characters, we focus on the 10 with the most total words spoken across seasons 1–25. This reduces clutter in the visualisations.

In [19]:
top10_characters = (
    df_clean
    .groupby("character")["word_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

print("Top 10 characters (by total words, seasons 1-25):")
for i, c in enumerate(top10_characters, 1):
    print(f"  {i}. {c}")

df_top10 = df_clean[df_clean["character"].isin(top10_characters)].copy()
print("\nFiltered dataset shape:", df_top10.shape)

Top 10 characters (by total words, seasons 1-25):
  1. Homer Simpson
  2. Marge Simpson
  3. Bart Simpson
  4. Lisa Simpson
  5. C. Montgomery Burns
  6. Moe Szyslak
  7. Seymour Skinner
  8. Ned Flanders
  9. Krusty the Clown
  10. Chief Wiggum

Filtered dataset shape: (76483, 11)


## 10. Build the four output datasets

Instead of one large line-level CSV, we produce **four focused CSVs**, each sized for its purpose:

| File | Granularity | Used for |
|---|---|---|
| `simpsons_lines.csv` | One row per dialogue line | Q3/Q4 histograms (word distribution) |
| `simpsons_character_totals.csv` | One row per character | Q1 ranking, Q5 sentence ranking |
| `simpsons_character_season.csv` | One row per character × season | Q2 evolution across seasons |
| `simpsons_character_episode.csv` | One row per character × episode | Q3/Q4 episode-level comparison |

This keeps each file small and avoids Altair's `MaxRowsError` without needing extra transformers.

In [20]:
# ── 10a. Line-level table (Q3 / Q4 histograms) ───────────────────────────
# Only the columns strictly needed: character, episode coordinates, word/sentence counts.
# title is included so Q4 charts can show the episode name in tooltips.
# timestamp_in_ms is included for Q4 word distribution timeline.
simpsons_lines = df_top10[[
    "episode_id",
    "season",
    "number_in_season",
    "title",
    "character",
    "word_count",
    "sentence_count",
    "timestamp_in_ms"
]].copy()

print("simpsons_lines shape:", simpsons_lines.shape)
simpsons_lines.head(3)

simpsons_lines shape: (76483, 8)


,episode_id,season,number_in_season,title,character,word_count,sentence_count,timestamp_in_ms
1,32,2,19,Lisa's Substitute,Lisa Simpson,3,2,856000
3,32,2,19,Lisa's Substitute,Lisa Simpson,5,1,864000
7,32,2,19,Lisa's Substitute,Bart Simpson,5,1,882000


In [21]:
# ── 10b. Character totals (Q1 word ranking, Q5 sentence ranking) ─────────
simpsons_character_totals = (
    df_top10
    .groupby("character", as_index=False)
    .agg(
        total_words=("word_count", "sum"),
        total_sentences=("sentence_count", "sum"),
        total_lines=("spoken_words", "count")
    )
    .sort_values("total_words", ascending=False)
    .reset_index(drop=True)
)

print("simpsons_character_totals shape:", simpsons_character_totals.shape)
simpsons_character_totals

simpsons_character_totals shape: (10, 4)


,character,total_words,total_sentences,total_lines
0,Homer Simpson,262982,48057,27106
1,Marge Simpson,120980,19998,12805
2,Bart Simpson,106317,19998,12700
3,Lisa Simpson,96830,16442,10522
4,C. Montgomery Burns,34314,5649,2921
5,Moe Szyslak,31021,5216,2658
6,Seymour Skinner,27936,4304,2370
7,Ned Flanders,22609,3502,2031
8,Krusty the Clown,19920,3273,1623
9,Chief Wiggum,19536,3294,1747


In [22]:
# ── 10c. Character × season totals (Q2 evolution) ───────────────────────
simpsons_character_season = (
    df_top10
    .groupby(["season", "character"], as_index=False)
    .agg(
        total_words=("word_count", "sum"),
        total_sentences=("sentence_count", "sum"),
        total_lines=("spoken_words", "count")
    )
    .sort_values(["season", "character"])
    .reset_index(drop=True)
)

print("simpsons_character_season shape:", simpsons_character_season.shape)
simpsons_character_season.head(6)

simpsons_character_season shape: (250, 5)


,season,character,total_words,total_sentences,total_lines
0,1,Bart Simpson,4868,1060,655
1,1,C. Montgomery Burns,646,117,52
2,1,Chief Wiggum,132,22,10
3,1,Homer Simpson,8146,1613,814
4,1,Krusty the Clown,374,65,43
5,1,Lisa Simpson,1871,367,271


In [23]:
# ── 10d. Character × episode totals + word share (Q3 / Q4 comparison) ───
simpsons_character_episode = (
    df_top10
    .groupby(["season", "number_in_season", "episode_id", "title", "character"], as_index=False)
    .agg(
        total_words=("word_count", "sum"),
        total_sentences=("sentence_count", "sum"),
        total_lines=("spoken_words", "count")
    )
)

# Word share within each episode (useful for normalised comparisons)
ep_total = (
    simpsons_character_episode
    .groupby(["season", "number_in_season", "episode_id"])["total_words"]
    .transform("sum")
)
simpsons_character_episode["episode_total_words"] = ep_total
simpsons_character_episode["word_share"] = (
    simpsons_character_episode["total_words"] / ep_total
).round(4)

simpsons_character_episode = (
    simpsons_character_episode
    .sort_values(["season", "number_in_season", "character"])
    .reset_index(drop=True)
)

print("simpsons_character_episode shape:", simpsons_character_episode.shape)
simpsons_character_episode.head(6)

simpsons_character_episode shape: (3876, 10)


,season,number_in_season,episode_id,title,character,total_words,total_sentences,total_lines,episode_total_words,word_share
0,1,1,1,Simpsons Roasting on an Open Fire,Bart Simpson,335,73,49,2004,0.1672
1,1,1,1,Simpsons Roasting on an Open Fire,C. Montgomery Burns,45,6,3,2004,0.0225
2,1,1,1,Simpsons Roasting on an Open Fire,Homer Simpson,900,212,113,2004,0.4491
3,1,1,1,Simpsons Roasting on an Open Fire,Lisa Simpson,222,26,21,2004,0.1108
4,1,1,1,Simpsons Roasting on an Open Fire,Marge Simpson,373,69,42,2004,0.1861
5,1,1,1,Simpsons Roasting on an Open Fire,Moe Szyslak,26,4,2,2004,0.0130


## 11. Final checks

We verify row counts, season coverage and absence of NaNs in key columns across all four datasets.

In [24]:
print("=" * 50)
print("Dataset summary")
print("=" * 50)
datasets = {
    "simpsons_lines": simpsons_lines,
    "simpsons_character_totals": simpsons_character_totals,
    "simpsons_character_season": simpsons_character_season,
    "simpsons_character_episode": simpsons_character_episode,
}
for name, ds in datasets.items():
    nulls = ds.isna().sum().sum()
    print(f"\n{name}")
    print(f"  Shape  : {ds.shape}")
    print(f"  Columns: {ds.columns.tolist()}")
    print(f"  NaNs   : {nulls}")

print("\nSeason coverage (lines):", sorted(simpsons_lines["season"].unique()))
print("Characters:", sorted(simpsons_lines["character"].unique()))

Dataset summary

simpsons_lines
  Shape  : (76483, 8)
  Columns: ['episode_id', 'season', 'number_in_season', 'title', 'character', 'word_count', 'sentence_count', 'timestamp_in_ms']
  NaNs   : 0

simpsons_character_totals
  Shape  : (10, 4)
  Columns: ['character', 'total_words', 'total_sentences', 'total_lines']
  NaNs   : 0

simpsons_character_season
  Shape  : (250, 5)
  Columns: ['season', 'character', 'total_words', 'total_sentences', 'total_lines']
  NaNs   : 0

simpsons_character_episode
  Shape  : (3876, 10)
  Columns: ['season', 'number_in_season', 'episode_id', 'title', 'character', 'total_words', 'total_sentences', 'total_lines', 'episode_total_words', 'word_share']
  NaNs   : 0

Season coverage (lines): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np

## 12. Save output CSVs

Four focused CSV files, each loaded independently by the Streamlit app:

- **`simpsons_lines.csv`** — line-level data for Q3/Q4 histograms.
- **`simpsons_character_totals.csv`** — one row per character for Q1 & Q5 rankings.
- **`simpsons_character_season.csv`** — one row per character × season for Q2.
- **`simpsons_character_episode.csv`** — one row per character × episode for Q3/Q4 comparisons.

In [25]:
simpsons_lines.to_csv("simpsons_lines.csv", index=False)
simpsons_character_totals.to_csv("simpsons_character_totals.csv", index=False)
simpsons_character_season.to_csv("simpsons_character_season.csv", index=False)
simpsons_character_episode.to_csv("simpsons_character_episode.csv", index=False)

print("Saved:")
for name, ds in datasets.items():
    print(f"  {name}.csv  →  {ds.shape[0]:,} rows × {ds.shape[1]} cols")

Saved:
  simpsons_lines.csv  →  76,483 rows × 8 cols
  simpsons_character_totals.csv  →  10 rows × 4 cols
  simpsons_character_season.csv  →  250 rows × 5 cols
  simpsons_character_episode.csv  →  3,876 rows × 10 cols
